In [2]:
import pandas as pd
import numpy as np

np.random.seed(42)

In [ ]:
# -- TABLE 1: Defining the Asset Metadata (Table 1: dim_aircraft)

aircraft_list = [f'N{i:03d}AL' for i in range(101, 111)]
dim_aircraft = pd.DataFrame({
    'tail_number': aircraft_list,
    'model': 'Alice_V1',
    'battery_capacity_kwh': 850, 
    'max_payload_lbs': 2500,
    'firmware_version': np.random.choice(['v2.1', 'v2.2'], 10)
})

dim_aircraft.to_csv('aviation_data/dim_aircraft.csv', index=False)

In [7]:
dim_aircraft.head()

,tail_number,model,battery_capacity_kwh,max_payload_lbs,firmware_version
0,N101AL,Alice_V1,850,2500,v2.1
1,N102AL,Alice_V1,850,2500,v2.1
2,N103AL,Alice_V1,850,2500,v2.1
3,N104AL,Alice_V1,850,2500,v2.1
4,N105AL,Alice_V1,850,2500,v2.2


In [8]:
# --- TABLE 2: Fact_Flight_Summary (Mission Metadata) 

num_flights = 100 

# 2. Mission Data
flight_summary = pd.DataFrame({
    'flight_id': [f'FL-{i:05d}' for i in range(1, num_flights + 1)],
    
    # Randomly assign an aircraft from our fleet (Link to Table 1)
    'tail_number': np.random.choice(dim_aircraft['tail_number'], num_flights),
    
    # Commercial Load: Cargo weight between 500 lbs and the 2500 lbs limit
    'payload_lbs': np.random.uniform(500, 2500, num_flights),
    
    # Environmental factors: Temperature affects battery efficiency significantly
    'ambient_temp_c': np.random.uniform(-5, 45, num_flights),
    
    # Strategic Revenue data for ROI modeling
    'ticket_revenue': np.random.uniform(3000, 7000, num_flights)
})


flight_summary.to_csv('aviation_data/flight_summary.csv', index=False)

In [9]:
flight_summary.head()

,flight_id,tail_number,payload_lbs,ambient_temp_c,ticket_revenue
0,FL-00001,N102AL,2349.286660,29.882094,5787.845849
1,FL-00002,N101AL,2338.231431,34.823589,6989.022139
2,FL-00003,N107AL,1005.980336,17.967340,6586.441052
3,FL-00004,N108AL,1890.822246,37.104571,5303.993671
4,FL-00005,N107AL,650.869094,33.445887,6669.582452


In [16]:
print(f"Dim_Aircraft Shape: {dim_aircraft.shape}")
print(f"Flight_Summary Shape: {flight_summary.shape}")

Dim_Aircraft Shape: (10, 5)
Flight_Summary Shape: (100, 5)


In [17]:
flight_summary.describe()

,payload_lbs,ambient_temp_c,ticket_revenue
count,100.000000,100.000000,100.000000
mean,1479.408265,18.532830,5213.820326
std,587.471162,13.791113,1076.893938
min,527.343930,-4.870249,3010.843598
25%,1005.206472,6.769575,4505.695178
50%,1456.256664,19.223880,5298.298534
75%,1999.322825,29.839328,6003.908894
max,2491.662750,44.312850,6990.770464


In [ ]:
# --- Table 3: Fact_Telemetry_HighFreq (w/ Official Spec Alignment)

telemetry_list = []
total_duration = 3000 
#50 minutes to reflect 250nm range potential

# 1. Each flight is an independent "experiment." We need to isolate each flight_id to 
    #apply its specific payload (weight) and ambient_temp (temperature)

for index, flight in fact_flight_summary.iterrows():
    f_id = flight['flight_id']
    payload = flight['payload_lbs']
    temp = flight['ambient_temp_c']

    # 2. Rules - define coefficients/penalties
    # Physics Factor: Heavy payload near 2500 lbs increases drain
    # Thermal Factor: Deviations from 25C increase cooling/heating power draw

    payload_factor = (payload / 2500) * 0.0006
    temp_factor = abs(temp - 25) * 0.00003

    current_soc = 100.0

    # 3. Use conditional logic (If/Else) that changes the aircraft's state based on the current time (by second)
    for second in range(total_duration):
        # --- Official Spec Implementation ---
        # 1. Climb Phase: 2,000 ft/min = 33.3 ft/sec
        if second < 450: # Climbing for 7.5 minutes
            phase = 'CLIMB'
            altitude = second * 33.3 
            phase_load = 0.025 # High Power Draw for dual 700kW motors
        
        # 2. Cruise Phase: Maintaining 15,000 ft (Typical regional altitude)
        elif second < 2500: 
            phase = 'CRUISE'
            altitude = 15000
            phase_load = 0.012 # Level flight efficiency
            
        # 3. Landing Phase: Descent
        else:
            phase = 'LANDING'
            altitude = 15000 - ((second - 2500) * 30)
            phase_load = 0.008

        # Calculate total drain and update SoC
        # Scaling for 50 min flight
            drain = (phase_load + payload_factor + temp_factor) / 10
            current_soc -= drain

telemetry_list.append({
            'flight_id': f_id,
            'timestamp_sec': second,
            'altitude_ft': max(0, altitude),
            'battery_soc_pct': round(current_soc, 2),
            'phase': phase
        })

fact_telemetry_high_freq = pd.DataFrame(telemetry_list)
fact_telemetry_high_freq.to_csv('aviation_data/fact_telemetry_high_freq.csv', index=False)